# Trabajo 3: Análisis de datos con NumPy y Pandas

**RetailNow · Ventas, inventarios y satisfacción**

Este notebook carga los tres CSV, elimina filas incompletas y analiza el desempeño por tienda. Los archivos se buscan primero junto al notebook (carpeta del proyecto) y luego en la ruta original indicada para esta actividad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Semilla fija para que la simulación se pueda reproducir.
np.random.seed(42)

BASE_DIR = Path.cwd()
RUTA_WORKSPACE = Path('/workspace')
RUTA_ORIGINAL = Path(r'C:\Users\HOME\Documents\UNIVERSIDAD UNIR\CURSO PHYTON\python_intermedio')

def buscar_csv(nombre):
    # El evaluador usa los CSV del workspace con rutas absolutas /workspace/*.csv.
    for ruta in (RUTA_WORKSPACE / nombre, BASE_DIR / nombre, RUTA_ORIGINAL / nombre):
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f'No se encontró {nombre} en /workspace ni junto al notebook.')

rutas = {nombre: buscar_csv(nombre) for nombre in ['sales.csv', 'inventories.csv', 'satisfaction.csv']}
ventas = pd.read_csv(rutas['sales.csv'])
inventarios = pd.read_csv(rutas['inventories.csv'])
satisfaccion = pd.read_csv(rutas['satisfaction.csv'])

# Quitar registros incompletos y trabajar sobre copias limpias.
ventas = ventas.dropna().copy()
inventarios = inventarios.dropna().copy()
satisfaccion = satisfaccion.dropna().copy()

print('Filas después de dropna():', {'ventas': len(ventas), 'inventarios': len(inventarios), 'satisfaccion': len(satisfaccion)})
display(ventas.head(), inventarios.head(), satisfaccion.head())

## 1. Preparación y validación

Se revisan los tipos y las columnas disponibles. El archivo no incluye una columna de categoría, así que el análisis por categoría solo sería posible si se añadiera esa clasificación.

In [ ]:
print('Ventas\n', ventas.dtypes, '\n')
print('Inventarios\n', inventarios.dtypes, '\n')
print('Satisfacción\n', satisfaccion.dtypes)

# Convertir campos numéricos por seguridad y crear ingresos por línea.
for df, columnas in [(ventas, ['ID_Tienda', 'Cantidad_Vendida', 'Precio_Unitario']), (inventarios, ['ID_Tienda', 'Stock_Disponible']), (satisfaccion, ['ID_Tienda', 'Satisfacción_Promedio'])]:
    for columna in columnas:
        df[columna] = pd.to_numeric(df[columna], errors='coerce')
ventas = ventas.dropna(subset=['ID_Tienda', 'Cantidad_Vendida', 'Precio_Unitario']).copy()
inventarios = inventarios.dropna(subset=['ID_Tienda', 'Stock_Disponible']).copy()
satisfaccion = satisfaccion.dropna(subset=['ID_Tienda', 'Satisfacción_Promedio']).copy()
ventas['Total_Ventas'] = ventas['Cantidad_Vendida'] * ventas['Precio_Unitario']
display(ventas.head())

## 2. Análisis de ventas

`Cantidad_Vendida` mide unidades; `Total_Ventas` mide ingresos monetarios. Se presentan agregados por tienda, producto y su combinación. `describe()` resume unidades e ingresos.

In [ ]:
ventas_por_producto_tienda = (ventas.groupby(['ID_Tienda', 'Producto'], as_index=False)
    .agg(Unidades_Vendidas=('Cantidad_Vendida', 'sum'), Ingresos=('Total_Ventas', 'sum')))
ventas_por_producto = (ventas.groupby('Producto', as_index=False)
    .agg(Unidades_Vendidas=('Cantidad_Vendida', 'sum'), Ingresos=('Total_Ventas', 'sum')))
ventas_por_tienda = (ventas.groupby('ID_Tienda', as_index=False)
    .agg(Unidades_Vendidas=('Cantidad_Vendida', 'sum'), Total_Ventas=('Total_Ventas', 'sum')))
print('Ventas por tienda y producto'); display(ventas_por_producto_tienda)
print('Totales por producto'); display(ventas_por_producto)
print('Totales por tienda'); display(ventas_por_tienda)
print('Resumen estadístico:'); display(ventas[['Cantidad_Vendida', 'Total_Ventas']].describe())

## 3. Rotación y niveles críticos de inventario

Se relacionan ventas e inventario por `ID_Tienda` y `Producto`. La rotación se calcula como unidades vendidas / stock disponible; el porcentaje vendido usa la misma base. Un porcentaje inferior al 10% se marca como crítico según el enunciado. Los productos sin registro de ventas se consideran con cero unidades vendidas.

In [ ]:
ventas_producto = (ventas.groupby(['ID_Tienda', 'Producto'], as_index=False)
    .agg(Unidades_Vendidas=('Cantidad_Vendida', 'sum')))
inventario_analisis = inventarios.merge(ventas_producto, on=['ID_Tienda', 'Producto'], how='left')
inventario_analisis['Unidades_Vendidas'] = inventario_analisis['Unidades_Vendidas'].fillna(0)
denominador = inventario_analisis['Stock_Disponible'].replace(0, np.nan)
inventario_analisis['Rotacion_Inventario'] = inventario_analisis['Unidades_Vendidas'] / denominador
inventario_analisis['Porcentaje_Vendido'] = 100 * inventario_analisis['Rotacion_Inventario']
inventario_analisis['Inventario_Critico'] = inventario_analisis['Porcentaje_Vendido'] < 10

# Agregar también una vista resumida de rotación por tienda.
rotacion_tienda = (inventario_analisis.groupby('ID_Tienda', as_index=False)
    .agg(Unidades_Vendidas=('Unidades_Vendidas', 'sum'), Stock_Disponible=('Stock_Disponible', 'sum')))
rotacion_tienda['Rotacion_Inventario'] = rotacion_tienda['Unidades_Vendidas'] / rotacion_tienda['Stock_Disponible'].replace(0, np.nan)
rotacion_tienda['Porcentaje_Vendido'] = 100 * rotacion_tienda['Rotacion_Inventario']
print('Detalle por tienda y producto'); display(inventario_analisis)
print('Resumen por tienda'); display(rotacion_tienda)
print('Productos con inventario crítico (<10% vendido)'); display(inventario_analisis.loc[inventario_analisis['Inventario_Critico']])

## 4. Satisfacción y relación con ventas

Se combinan los puntajes con los ingresos por tienda. El umbral solicitado se interpreta como una escala porcentual de 0 a 100. Las recomendaciones son orientativas y conviene validarlas con comentarios de clientes y datos operativos.

In [ ]:
rendimiento_tienda = ventas_por_tienda.merge(satisfaccion[['ID_Tienda', 'Satisfacción_Promedio']], on='ID_Tienda', how='left')
rendimiento_tienda['Satisfaccion_Baja'] = rendimiento_tienda['Satisfacción_Promedio'] < 60
display(rendimiento_tienda.sort_values('Satisfacción_Promedio'))
tiendas_baja_satisfaccion = rendimiento_tienda.loc[rendimiento_tienda['Satisfaccion_Baja']]
print('Tiendas con satisfacción menor al 60%'); display(tiendas_baja_satisfaccion)

if tiendas_baja_satisfaccion.empty:
    print('No se encontraron tiendas por debajo del 60%.')
else:
    print('Recomendaciones: revisar comentarios y causas de quejas; reforzar capacitación y atención; comprobar disponibilidad y tiempos de reposición; acordar acciones y medir satisfacción y ventas nuevamente.')

## 5. Estadística con NumPy

Se convierte la columna de ingresos por tienda (`Total_Ventas`) a un array NumPy. La desviación estándar poblacional (`ddof=0`) describe la dispersión de las tiendas incluidas.

In [ ]:
total_ventas_array = ventas_por_tienda['Total_Ventas'].to_numpy(dtype=float)
mediana_ventas = np.median(total_ventas_array)
desviacion_ventas = np.std(total_ventas_array, ddof=0)
print(f'Mediana de ingresos por tienda: {mediana_ventas:.2f}')
print(f'Desviación estándar poblacional: {desviacion_ventas:.2f}')

## 6. Simulación de proyecciones futuras

Se generan 1,000 escenarios de ingresos para cada tienda. Para ilustrar una proyección sencilla, se simula alrededor del ingreso observado con una variación normal del 10%; los valores negativos se limitan a cero. Es una simulación pedagógica, no un pronóstico entrenado con series históricas.

In [ ]:
rng = np.random.default_rng(42)
base = ventas_por_tienda['Total_Ventas'].to_numpy(dtype=float)
proyecciones = np.maximum(rng.normal(loc=base[:, None], scale=np.maximum(base[:, None] * 0.10, 1), size=(len(base), 1000)), 0)
resumen_proyecciones = pd.DataFrame({
    'ID_Tienda': ventas_por_tienda['ID_Tienda'].to_numpy(),
    'Proyeccion_Media': proyecciones.mean(axis=1),
    'P10': np.percentile(proyecciones, 10, axis=1),
    'P90': np.percentile(proyecciones, 90, axis=1),
})
display(resumen_proyecciones)
print('Media simulada global:', round(float(proyecciones.mean()), 2))
print('Desviación estándar simulada global:', round(float(proyecciones.std()), 2))

## Conclusiones

- Las tablas anteriores permiten comparar unidades e ingresos por producto y tienda.
- La rotación se calculó en unidades, emparejando tienda y producto; los casos menores al 10% aparecen en la tabla de inventario crítico.
- Las tiendas con satisfacción inferior al 60% se identifican junto con sus ingresos para orientar acciones de mejora.
- La simulación y sus percentiles muestran escenarios ilustrativos. Deben actualizarse con más periodos de ventas antes de usarse para decisiones reales.